# Projeto II - Análise e Treino do LDR

Notebook de apoio para apresentação e análise do projeto `Estimativa Reacional Regressiva Perceptiva Aplicada Sobre Iluminação Residencial Dinâmica`.

Este notebook usa apenas o dataset real coletado no Wokwi e mostra os gráficos principais usados para justificar a calibração do sensor.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'wokwi.toml').exists():
            return candidate
    raise FileNotFoundError('Nao foi possivel localizar a raiz do projeto a partir do diretorio atual.')

PROJECT_ROOT = find_project_root(Path.cwd())
ARTIFACTS_DIR = PROJECT_ROOT / 'training' / 'artifacts'
DATASET_PATH = ARTIFACTS_DIR / 'wokwi_training_dataset.csv'
ADC_MAX = 4095.0

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset não encontrado: {DATASET_PATH}\n'
        'Rode antes: python .\\training\\import_wokwi_samples.py'
    )

df = pd.read_csv(DATASET_PATH)
if 'log_inverse_ratio_feature' not in df.columns:
    adc = df['adc_raw'].clip(lower=1.0, upper=ADC_MAX - 1.0)
    df['log_inverse_ratio_feature'] = np.log((ADC_MAX - adc) / adc)
if 'log_lux_referencia' not in df.columns:
    df['log_lux_referencia'] = np.log1p(df['lux_referencia'])

print('Raiz do projeto:', PROJECT_ROOT)
print('Dataset carregado:', DATASET_PATH)
print('Amostras:', len(df))
print('Colunas:', ', '.join(df.columns))


## Visão Geral dos Dados

As amostras foram coletadas no Wokwi associando o valor de iluminação do slider com a leitura bruta do ADC no ESP32-S3.

In [ ]:
display(df.head())
display(df.describe(include='all').transpose())

## Modelo de Regressão Usado para Análise

Para explicar o comportamento do sensor, usamos a feature logarítmica:

`log((4095 - adc_raw) / adc_raw)`

e ajustamos uma regressão linear para prever `log(lux)`.

In [ ]:
feature = df[['log_inverse_ratio_feature']].to_numpy(dtype=np.float64)
target = np.log(np.maximum(df['lux_referencia'].to_numpy(dtype=np.float64), 1e-9))

model = LinearRegression()
model.fit(feature, target)

pred_log_lux = model.predict(feature)
pred_lux = np.exp(pred_log_lux)

rmse = root_mean_squared_error(df['lux_referencia'], pred_lux)
mape = mean_absolute_percentage_error(df['lux_referencia'], pred_lux) * 100.0

results = df.copy()
results['lux_previsto'] = pred_lux
results['erro_absoluto'] = np.abs(results['lux_previsto'] - results['lux_referencia'])
results['erro_percentual'] = (
    results['erro_absoluto'] / results['lux_referencia'].clip(lower=1e-9)
) * 100.0

print(f'Peso: {model.coef_[0]:.6f}')
print(f'Bias: {model.intercept_:.6f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAPE: {mape:.2f}%')

## Gráficos para Apresentação

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].semilogx(df['lux_referencia'], df['adc_raw'], marker='o', color='#d95f02')
axes[0, 0].set_title('Lux de Referência x ADC Raw')
axes[0, 0].set_xlabel('Lux de referência')
axes[0, 0].set_ylabel('ADC raw')

axes[0, 1].plot(df['adc_raw'], df['log_inverse_ratio_feature'], marker='o', color='#1b9e77')
axes[0, 1].set_title('ADC Raw x Feature Logarítmica')
axes[0, 1].set_xlabel('ADC raw')
axes[0, 1].set_ylabel('log((4095 - adc) / adc)')

sorted_idx = np.argsort(df['log_inverse_ratio_feature'].to_numpy())
axes[1, 0].scatter(df['log_inverse_ratio_feature'], np.log(df['lux_referencia']), color='#7570b3')
axes[1, 0].plot(
    df['log_inverse_ratio_feature'].to_numpy()[sorted_idx],
    pred_log_lux[sorted_idx],
    color='#e7298a',
    linewidth=2,
)
axes[1, 0].set_title('Feature Logarítmica x log(Lux)')
axes[1, 0].set_xlabel('Feature logarítmica')
axes[1, 0].set_ylabel('log(lux)')

axes[1, 1].loglog(df['lux_referencia'], df['lux_previsto'], marker='o', linestyle='none', color='#66a61e')
min_lux = float(df['lux_referencia'].min())
max_lux = float(df['lux_referencia'].max())
axes[1, 1].plot([min_lux, max_lux], [min_lux, max_lux], '--', color='black', linewidth=1)
axes[1, 1].set_title('Lux Real x Lux Previsto')
axes[1, 1].set_xlabel('Lux real')
axes[1, 1].set_ylabel('Lux previsto')

fig.suptitle('Análise do Sensor LDR e do Modelo', fontsize=16)
fig.tight_layout()
plt.show()

## Tabela de Validação

A tabela abaixo ajuda a mostrar, em sala ou no relatório, como o valor previsto acompanha o valor de referência.

In [ ]:
validation_view = results[
    [
        'lux_referencia',
        'adc_raw',
        'lux_previsto',
        'erro_absoluto',
        'erro_percentual',
    ]
].copy()

validation_view['lux_previsto'] = validation_view['lux_previsto'].round(2)
validation_view['erro_absoluto'] = validation_view['erro_absoluto'].round(2)
validation_view['erro_percentual'] = validation_view['erro_percentual'].round(2)

display(validation_view)

## Conclusão

Os gráficos mostram que a relação entre iluminância e leitura do sensor não é linear direta no espaço original. A transformação logarítmica melhora a modelagem e permite obter uma estimativa de lux consistente para uso embarcado no ESP32-S3 com TensorFlow Lite Micro.